# COVID-19 Chest X-ray Classification using CNN, Deep CNN, ResNet50 and VGG16
Complete end-to-end deep learning project for multi-class classification of chest X-ray images.

## Objective
Build and compare multiple deep learning models for COVID-19 detection from chest X-ray images.

Models Included:
- Basic CNN
- Deep CNN
- ResNet50 (Transfer Learning)
- VGG16 (Transfer Learning)
- Streamlit Web App

## Project Description
Mini Project: COVID-19 Detection from Chest X-rays using CNN
Objective
A healthcare startup aims to accelerate and improve COVID-19 diagnosis using deep learning technologies. The current testing procedures are time-consuming and rely heavily on manual radiological examination, which limits scalability in pandemic situations.
As a data scientist, your task is to build a Convolutional Neural Network (CNN) model that can automatically detect COVID-19 from chest X-ray images. This solution can help hospitals:
Reduce diagnosis time.
Minimize burden on radiologists.
Scale testing across regions with limited radiological expertise.
This project allows you to apply deep learning concepts, understand how CNNs process image data, and build real-world applications that can assist the medical community.
Dataset
You will use the dataset provided in the following Kaggle project:
https://www.kaggle.com/datasets/pranavraikokte/covid19-image-dataset
The dataset contains:
COVID-19 chest X-ray images
Normal (no disease) chest X-ray images
Viral Pneumonia chest X-ray images
Data Dictionary
Each image is labelled into one of the following classes:
COVID-19 – Confirmed COVID infection.
Viral Pneumonia – Non-COVID lung infection.
Normal – No visible lung abnormality.
Tasks
1. Data Loading and Exploration
Import necessary libraries (os, cv2, matplotlib, tensorflow, keras etc.)
Import Covid19 dataset from Kaggle into colab using Kaggle API.
Load images from different folders and label them
Resize images to a fixed shape (e.g 128x128 or 224x224)
Display a few sample images from each class
Print dataset size per class
2. Data Preprocessing
Normalize pixel values (scale from 0–255 to 0–1)
Encode class labels using one-hot encoding or LabelEncoder
Split the data into training, validation and test sets (e.g 80-20%)
3. Exploratory Data Analysis (EDA)
Visualize class distribution using bar plots
Plot sample images with their class names
Observe patterns in data
4. CNN Model Building
Create and train multiple CNN architectures:
Model 1: Basic CNN
Conv2D -> MaxPooling -> Flatten -> Dense
Model 2: Transfer Learning
Use pre-trained models like VGG16, ResNet50 etc
Fine-tune last few layers on the COVID dataset
Model 3: Transfer Learning + Data Augmentation
Do the data Augmentation using ImageDataGenerator.
The use the pretrained models to get the prediction.
5. Model Evaluation
Evaluate each model using:
Accuracy on test set
Confusion matrix
Classification report (Precision, Recall, F1-score)
ROC-AUC score (binary or per class if multi-class)
Also check for:
Overfitting (training vs validation loss/accuracy plots)
Number of trainable parameters
6. Handle Class Imbalance
Analyze class imbalance using visual plots
Apply techniques like:
Oversampling using data augmentation
Class weights in model.fit()
7. Model Tuning
Use techniques like:
Early stopping
Hyperparameter tuning (number of filters, dropout rate, optimizer etc.)
Use Keras Tuner for deeper optimization.
8. Model Comparison Table
9. Build a web app using Streamlit that:
Takes a chest X-ray image as input
Displays prediction

In [1]:
# Install required libraries
import sys

print(f"Python version: {sys.version}")

if sys.version_info >= (3, 14):
    raise RuntimeError(
        "TensorFlow is not supported on Python 3.14+. ",
        "Please switch the notebook kernel to Python 3.11 or 3.12."
    )

%pip install -q kaggle keras-tuner streamlit opencv-python matplotlib seaborn scikit-learn pandas pillow tensorboard
%pip install -q --upgrade --force-reinstall tensorflow-cpu==2.15.0


Python version: 3.12.0 | packaged by Anaconda, Inc. | (main, Oct  2 2023, 17:20:38) [MSC v.1916 64 bit (AMD64)]
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow-cpu==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow-cpu==2.15.0


In [2]:
# Import Libraries
import os
# Clear any invalid matplotlib backend settings from the environment
os.environ.pop('MPLBACKEND', None)

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    Dropout, BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

import keras_tuner as kt

print("TensorFlow Version:", tf.__version__)


TensorFlow Version: 2.21.0


## Dataset Location

In [3]:
# This notebook uses the local dataset archive included in the repository.
# Update the dataset path in the next cell if your files are stored somewhere else.


In [4]:
# Dataset Path
# This notebook uses the local dataset archive included in the repository.
# Update the path below if your dataset is stored elsewhere.

dataset_path = os.path.join(os.getcwd(), 'archive (2)', 'Covid19-dataset', 'train')

if not os.path.isdir(dataset_path):
    raise FileNotFoundError(
        f"Dataset path not found: {dataset_path}.\n" \
        "Please extract the dataset archive into the repository or update the path."
    )

classes = ['Covid', 'Normal', 'Viral Pneumonia']

IMG_SIZE = 224

## Data Loading and Preprocessing

In [5]:
# Load Images
images = []
labels = []

for label in classes:
    folder_path = os.path.join(dataset_path, label)
    
    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        
        try:
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            
            images.append(img)
            labels.append(label)
        except:
            pass

X = np.array(images)
y = np.array(labels)

print("Dataset Shape:", X.shape)
print("Labels Shape:", y.shape)


Dataset Shape: (251, 224, 224, 3)
Labels Shape: (251,)


In [6]:
# Display Sample Images
plt.figure(figsize=(12,6))

for i, cls in enumerate(classes):
    idx = np.where(y == cls)[0][0]
    
    plt.subplot(1,3,i+1)
    plt.imshow(X[idx])
    plt.title(cls)
    plt.axis('off')

plt.show()


C:\Users\Loq\AppData\Local\Temp\ipykernel_13612\2281932085.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Class Distribution
class_counts = pd.Series(y).value_counts()

plt.figure(figsize=(6,4))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

print(class_counts)


Covid              111
Normal              70
Viral Pneumonia     70
Name: count, dtype: int64


C:\Users\Loq\AppData\Local\Temp\ipykernel_13612\379145556.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Normalize Images
X = X / 255.0

# Encode Labels
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# One Hot Encoding
y_cat = to_categorical(y_encoded)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_cat
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


Train Shape: (200, 224, 224, 3)
Test Shape: (51, 224, 224, 3)


## Data Augmentation

In [9]:
train_datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

train_datagen.fit(X_train)


## Model 1: Basic CNN

In [10]:
basic_cnn = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(2,2),
    
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Flatten(),
    
    Dense(128, activation='relu'),
    Dropout(0.5),
    
    Dense(3, activation='softmax')
])

basic_cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

basic_cnn.summary()


d:\Ankit\IITGAIML\Deployment\Mini Project COVID-19 Detection from Chest X-rays using CNN\myenv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,907,779 (91.20 MB)

 Trainable params: 23,907,779 (91.20 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history_basic = basic_cnn.fit(
    train_datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_test, y_test),
    epochs=20,
    callbacks=[early_stop]
)


Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 466ms/step - accuracy: 0.3350 - loss: 5.5291 - val_accuracy: 0.4510 - val_loss: 0.9339
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 354ms/step - accuracy: 0.5300 - loss: 1.0304 - val_accuracy: 0.7647 - val_loss: 0.8410
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 348ms/step - accuracy: 0.6400 - loss: 0.8081 - val_accuracy: 0.8431 - val_loss: 0.5308
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 384ms/step - accuracy: 0.7000 - loss: 0.6699 - val_accuracy: 0.8235 - val_loss: 0.4455
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 365ms/step - accuracy: 0.7500 - loss: 0.5999 - val_accuracy: 0.7451 - val_loss: 0.4570
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 327ms/step - accuracy: 0.7700 - loss: 0.6230 - val_accuracy: 0.8039 - val_loss: 0.5140
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 334ms/step - accuracy: 0.7300 - loss: 0.6542 - val_accuracy: 0.7843 - val_loss: 0.3812
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 352ms/step - accuracy: 0.7700 - loss: 0.6010 - val_accuracy: 0.8627 - val_loss:

## Model 2: Deep CNN

In [12]:
deep_cnn = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),
    
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    
    Conv2D(256, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    
    Flatten(),
    
    Dense(256, activation='relu'),
    Dropout(0.5),
    
    Dense(128, activation='relu'),
    Dropout(0.3),
    
    Dense(3, activation='softmax')
])

deep_cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

deep_cnn.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 24, 24, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,861,059 (37.62 MB)

 Trainable params: 9,860,099 (37.61 MB)

 Non-trainable params: 960 (3.75 KB)

In [13]:
history_deep = deep_cnn.fit(
    train_datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_test, y_test),
    epochs=25,
    callbacks=[early_stop]
)


Epoch 1/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 812ms/step - accuracy: 0.5500 - loss: 6.6673 - val_accuracy: 0.2745 - val_loss: 3.8571
Epoch 2/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 737ms/step - accuracy: 0.6600 - loss: 6.3121 - val_accuracy: 0.2745 - val_loss: 8.9029
Epoch 3/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 834ms/step - accuracy: 0.6600 - loss: 6.6042 - val_accuracy: 0.5098 - val_loss: 1.3055
Epoch 4/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 706ms/step - accuracy: 0.7400 - loss: 6.0338 - val_accuracy: 0.6275 - val_loss: 1.1020
Epoch 5/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 721ms/step - accuracy: 0.7550 - loss: 3.5594 - val_accuracy: 0.4706 - val_loss: 3.0059


## Model 3: ResNet50 Transfer Learning

In [14]:
base_resnet = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_resnet.layers:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_resnet.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(3, activation='softmax')(x)

resnet_model = Model(inputs=base_resnet.input, outputs=output)

resnet_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

resnet_model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_2[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,113,027 (91.98 MB)

 Trainable params: 525,315 (2.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [15]:
history_resnet = resnet_model.fit(
    train_datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_test, y_test),
    epochs=15,
    callbacks=[early_stop]
)


Epoch 1/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.3900 - loss: 1.3967 - val_accuracy: 0.5294 - val_loss: 1.0289
Epoch 2/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 842ms/step - accuracy: 0.4200 - loss: 1.2692 - val_accuracy: 0.5490 - val_loss: 0.9399
Epoch 3/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 858ms/step - accuracy: 0.4900 - loss: 1.0521 - val_accuracy: 0.6863 - val_loss: 0.8981
Epoch 4/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 944ms/step - accuracy: 0.4950 - loss: 1.0444 - val_accuracy: 0.6275 - val_loss: 0.8737
Epoch 5/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 900ms/step - accuracy: 0.5150 - loss: 0.9794 - val_accuracy: 0.6667 - val_loss: 0.8528


## Model 4: VGG16 Transfer Learning

In [16]:
base_vgg = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_vgg.layers:
    layer.trainable = False

x = Flatten()(base_vgg.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(3, activation='softmax')(x)

vgg_model = Model(inputs=base_vgg.input, outputs=output)

vgg_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

vgg_model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,138,243 (80.64 MB)

 Trainable params: 6,423,555 (24.50 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [17]:
history_vgg = vgg_model.fit(
    train_datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_test, y_test),
    epochs=15,
    callbacks=[early_stop]
)


Epoch 1/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 19s 3s/step - accuracy: 0.5350 - loss: 2.7064 - val_accuracy: 0.8235 - val_loss: 0.4222
Epoch 2/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 16s 2s/step - accuracy: 0.6800 - loss: 1.7280 - val_accuracy: 0.8235 - val_loss: 0.4745
Epoch 3/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 16s 3s/step - accuracy: 0.7150 - loss: 1.1306 - val_accuracy: 0.6667 - val_loss: 1.1966
Epoch 4/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 16s 2s/step - accuracy: 0.7750 - loss: 0.7738 - val_accuracy: 0.8824 - val_loss: 0.2902
Epoch 5/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - accuracy: 0.8300 - loss: 0.3717 - val_accuracy: 0.8235 - val_loss: 0.5093


## Evaluation Functions

In [18]:
def evaluate_model(model, X_test, y_test, model_name):
    
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true = np.argmax(y_test, axis=1)

    print(f"\n===== {model_name} =====")
    
    print(classification_report(y_true, y_pred_classes))
    
    cm = confusion_matrix(y_true, y_pred_classes)
    
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
    
    roc_score = roc_auc_score(y_test, y_pred, multi_class='ovr')
    print("ROC-AUC Score:", roc_score)

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    
    return acc, roc_score


In [19]:
basic_acc, basic_roc = evaluate_model(
    basic_cnn, X_test, y_test, "Basic CNN"
)

deep_acc, deep_roc = evaluate_model(
    deep_cnn, X_test, y_test, "Deep CNN"
)

resnet_acc, resnet_roc = evaluate_model(
    resnet_model, X_test, y_test, "ResNet50"
)

vgg_acc, vgg_roc = evaluate_model(
    vgg_model, X_test, y_test, "VGG16"
)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step

===== Basic CNN =====
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        23
           1       0.83      0.71      0.77        14
           2       0.80      0.86      0.83        14

    accuracy                           0.88        51
   macro avg       0.86      0.86      0.86        51
weighted avg       0.88      0.88      0.88        51

ROC-AUC Score: 0.9755469755469756


C:\Users\Loq\AppData\Local\Temp\ipykernel_13612\3319280007.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 222ms/step

===== Deep CNN =====
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        23
           1       0.27      1.00      0.43        14
           2       0.00      0.00      0.00        14

    accuracy                           0.27        51
   macro avg       0.09      0.33      0.14        51
weighted avg       0.08      0.27      0.12        51

ROC-AUC Score: 0.7769151138716356


d:\Ankit\IITGAIML\Deployment\Mini Project COVID-19 Detection from Chest X-rays using CNN\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Ankit\IITGAIML\Deployment\Mini Project COVID-19 Detection from Chest X-rays using CNN\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Ankit\IITGAIML\Deployment\Mini Project COVID-19 Detection from Chest X-rays using CNN\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being

1/2 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/stepWARNING:tensorflow:6 out of the last 6 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x000001B56C07D4E0> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step

===== ResNet50 =====
              precision    recall  f1-score   support

           0       1.00      0.57      0.72        23
           1       0.00      0.00      0.00        14
           2      

d:\Ankit\IITGAIML\Deployment\Mini Project COVID-19 Detection from Chest X-rays using CNN\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Ankit\IITGAIML\Deployment\Mini Project COVID-19 Detection from Chest X-rays using CNN\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Ankit\IITGAIML\Deployment\Mini Project COVID-19 Detection from Chest X-rays using CNN\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being

2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step

===== VGG16 =====
              precision    recall  f1-score   support

           0       0.88      0.91      0.89        23
           1       0.82      0.64      0.72        14
           2       0.75      0.86      0.80        14

    accuracy                           0.82        51
   macro avg       0.81      0.80      0.80        51
weighted avg       0.83      0.82      0.82        51

ROC-AUC Score: 0.9560740865088692


C:\Users\Loq\AppData\Local\Temp\ipykernel_13612\3319280007.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Training vs Validation Accuracy

In [20]:
def plot_history(history, title):

    plt.figure(figsize=(8,5))
    
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    
    plt.title(title)
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()

plot_history(history_basic, 'Basic CNN Accuracy')
plot_history(history_deep, 'Deep CNN Accuracy')
plot_history(history_resnet, 'ResNet50 Accuracy')
plot_history(history_vgg, 'VGG16 Accuracy')


C:\Users\Loq\AppData\Local\Temp\ipykernel_13612\2154197222.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Handle Class Imbalance using Class Weights

In [21]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_encoded),
    y=y_encoded
)

class_weights = dict(enumerate(class_weights))

print(class_weights)


{0: np.float64(0.7537537537537538), 1: np.float64(1.1952380952380952), 2: np.float64(1.1952380952380952)}


## Hyperparameter Tuning using Keras Tuner

In [22]:
def build_tuner_model(hp):

    model = Sequential()

    model.add(
        Conv2D(
            filters=hp.Int('filters', 32, 128, step=32),
            kernel_size=(3,3),
            activation='relu',
            input_shape=(224,224,3)
        )
    )

    model.add(MaxPooling2D(2,2))
    model.add(Flatten())

    model.add(
        Dense(
            units=hp.Int('units', 64, 256, step=64),
            activation='relu'
        )
    )

    model.add(Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1)))

    model.add(Dense(3, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


tuner = kt.RandomSearch(
    build_tuner_model,
    objective='val_accuracy',
    max_trials=5,
    directory='keras_tuner_dir',
    project_name='covid_xray'
)

tuner.search(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=5
)

best_model = tuner.get_best_models(num_models=1)[0]


Reloading Tuner from keras_tuner_dir\covid_xray\tuner0.json



d:\Ankit\IITGAIML\Deployment\Mini Project COVID-19 Detection from Chest X-rays using CNN\myenv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
d:\Ankit\IITGAIML\Deployment\Mini Project COVID-19 Detection from Chest X-rays using CNN\myenv\Lib\site-packages\keras\src\saving\saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [23]:
# Diagnostic cell: capture full traceback, shapes and memory
import sys, traceback, gc, os
import psutil
import numpy as np
print('Python:', sys.version)
# Inspect training arrays if available
try:
    print('X_train shape, dtype:', getattr(X_train, 'shape', None), getattr(X_train, 'dtype', None))
    print('y_train shape, dtype:', getattr(y_train, 'shape', None), getattr(y_train, 'dtype', None))
except Exception as e:
    print('Could not inspect X_train/y_train:', e)
# Process memory usage (MB)
try:
    proc = psutil.Process()
    print('Process RSS (MB):', proc.memory_info().rss / 1024**2)
except Exception as e:
    print('psutil error:', e)
# Attempt a short tuner.search to capture the error with minimal work
try:
    print('Starting short tuner.search (epochs=1, batch_size=8)')
    tuner.search(X_train, y_train, validation_data=(X_test, y_test), epochs=1, batch_size=8)
except Exception:
    print('Tuner raised an exception:')
    print(traceback.format_exc())
    gc.collect()
    print('Diagnostic complete')

Python: 3.12.0 | packaged by Anaconda, Inc. | (main, Oct  2 2023, 17:20:38) [MSC v.1916 64 bit (AMD64)]
X_train shape, dtype: (200, 224, 224, 3) float64
y_train shape, dtype: (200, 3) float64
Process RSS (MB): 5161.23046875
Starting short tuner.search (epochs=1, batch_size=8)


## Model Comparison Table

In [26]:
def detect_overfitting(history, threshold=0.03):
    train_acc = history.history.get('accuracy', [0])[-1]
    val_acc = history.history.get('val_accuracy', [0])[-1]
    return 'Yes' if (train_acc - val_acc) > threshold else 'No'

comparison_df = pd.DataFrame({
    'Model': ['CNN Basic', 'Deep CNN', 'ResNet50', 'VGG16'],
    'Test Accuracy': [
        basic_acc,
        deep_acc,
        resnet_acc,
        vgg_acc
    ],
    'Train Accuracy': [
        history_basic.history['accuracy'][-1],
        history_deep.history['accuracy'][-1],
        history_resnet.history['accuracy'][-1],
        history_vgg.history['accuracy'][-1]
    ],
    'ROC-AUC': [
        basic_roc,
        deep_roc,
        resnet_roc,
        vgg_roc
    ],
    'Overfitting(Y/N)': [
        detect_overfitting(history_basic),
        detect_overfitting(history_deep),
        detect_overfitting(history_resnet),
        detect_overfitting(history_vgg)
    ]
})

comparison_df


,Model,Test Accuracy,Train Accuracy,ROC-AUC,Overfitting(Y/N)
0,CNN Basic,0.882353,0.815,0.975547,No
1,Deep CNN,0.274510,0.755,0.776915,Yes
2,ResNet50,0.529412,0.515,0.850106,No
3,VGG16,0.823529,0.830,0.956074,No


## Save Best Model

In [ ]:
resnet_model.save('best_covid_model.h5')
print("Model Saved Successfully")

best_model_name = comparison_df.loc[comparison_df['Test Accuracy'].idxmax(), 'Model']
print(f"Best Model based on test accuracy: {best_model_name}")


Model Saved Successfully
